In [1]:
import pdb
import sqlite3
import logging
import datetime as dt

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
def dict_factory(cursor, row):
    fields = [column[0] for column in cursor.description]
    return {key: value for key, value in zip(fields, row)}

In [4]:
def get_time_id(cursor, date_str):
    if not date_str:
        return None

    row = cursor.execute(
        "select id from dim_time where dt = ?", (date_str[:13],)
    ).fetchone()

    return row["id"] if row else None

In [5]:
def create_dim_time(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_time (
        id integer primary key, -- automatically auto increments
        dt datetime unique,
        year int,
        month int,
        day int,
        hour int,
        quarter int,
        day_name text,
        day_name_abbr text,
        month_name text,
        month_name_abbr text,
        is_weekend int,
        day_period text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [6]:
def create_dim_product(conn: sqlite3.Connection):
    sql = """
    -- Representa: Products, Categories e Suppliers
    -- Três opções de gerenciamento dos valores:
    -- primeira inserção é a que fica
    -- última inserção é a que fica (update)
    -- guardamos todos os estados (surrogate obrigatória)
    -- (product_id, product_unit_price) => histórico de variação
    -- dos preços do produto
    create table if not exists dim_product (
        id integer primary key, -- automatically auto increments
        product_id int unique,  -- chave natural
        product_name text,
        category_name text,
        category_description text,
        supplier_company text,
        supplier_city text,
        supplier_region text,
        supplier_country text,
        product_unit_price numeric,
        product_quantity_per_unit numeric,
        product_discontinued boolean
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [7]:
def create_dim_customer(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_customer (
        id integer primary key, -- automatically auto increments
        customer_id text unique,       -- chave natural
        company_name text,
        contact_name text,
        city text,
        region text,
        country text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [8]:
def create_dim_employee(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_employee (
        id integer primary key, -- automatically auto increments
        employee_id int unique, -- chave natural
        full_name text,
        title text,
        birth_date date,
        hire_date date,
        city text,
        region text,
        country text,
        reports_to_name text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [9]:
def create_dim_shipper(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_shipper (
        id integer primary key, -- automatically auto increments
        shipper_id int unique,  -- chave natural
        company_name text,
        phone text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [10]:
def create_dim_ship_location(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_ship_location (
        id integer primary key, -- automatically auto increments
        city text,
        region text,
        country text,
        unique (city, region, country)
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [11]:
def create_fact_sales(conn: sqlite3.Connection):
    sql = """
    -- Representa: Orders e Order Details
    create table if not exists fact_sales (
        id integer primary key, -- automatically auto increments
        order_id int,    -- chave natural
        -- Chaves Estrangeiras
        order_time_id int references dim_time(id),
        required_time_id int references dim_time(id),
        shipped_time_id int references dim_time(id) null,
        customer_id int references dim_customer(id),
        product_id int references dim_product(id),  -- chave natural
        employee_id int references dim_employee(id),
        shipper_id int references dim_shipper(id),
        ship_location_id int references dim_ship_location(id),
        -- Metricas
        unit_price_sold numeric,
        quantity int,
        discount numeric,
        revenue_net numeric, -- (Price * Qty) * (1 - Discount)
        freight_apportioned numeric, -- granularidade do campo diferente da granularidade da dimensão
        unique(order_id, product_id)
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [12]:
def insert_dim_time(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
    dt_range: tuple[dt.datetime, dt.datetime] = None
):
    def fetch_start_end_dates():
        sql_dates = """
        select min(datetime(OrderDate)) as dt_start,
               max(datetime(OrderDate)) as dt_end
        from Orders
        """
        cur = src_conn.cursor()
        res = cur.execute(sql_dates).fetchone()
        return res["dt_start"], res["dt_end"]

    def calculate_day_period(hour: int):
        if 6 <= hour and hour <= 11:
            return "Morning"
        elif 12 <= hour and hour <= 17:
            return "Afternoon"
        elif 18 <= hour and hour <= 23:
            return "Night"
        else:
            return "Midnight"

    if not dt_range:
        dt_start, dt_end = fetch_start_end_dates()
        dt_start = dt.datetime.strptime(dt_start, "%Y-%m-%d %H:%M:%S")
        dt_end = dt.datetime.strptime(dt_end, "%Y-%m-%d %H:%M:%S")
    else:
        dt_start, dt_end = dt_range

    dt_aux = dt_start.replace(hour=0, minute=0, second=0)
    values = []
    i = 1
    while dt_aux <= dt_end:
        values.append(
            {
                "dt": dt_aux.strftime('%Y-%m-%d %H'),
                "year": dt_aux.year,
                "month": dt_aux.month,
                "day": dt_aux.day,
                "hour": dt_aux.hour,
                "quarter": (dt_aux.month - 1) // 3 + 1,
                "day_name": dt_aux.strftime("%A"),
                "day_name_abbr": dt_aux.strftime("%a"),
                "month_name": dt_aux.strftime("%B"),
                "month_name_abbr": dt_aux.strftime("%b"),
                "is_weekend": dt_aux.weekday() > 4,
                "day_period": calculate_day_period(dt_aux.hour),
            }
        )
        if len(values) % 10_000 == 0:
            logger.info(f"Inserted {i * len(values)} date times")
            i += 1
            cur = dst_conn.cursor()
            cur.executemany("""
                insert into dim_time (
                    dt, year, month, day, hour, quarter, day_name, day_name_abbr, month_name, month_name_abbr, is_weekend, day_period
                ) values (
                    :dt, :year, :month, :day, :hour, :quarter, :day_name, :day_name_abbr, :month_name, :month_name_abbr, :is_weekend, :day_period
                )
                """,
                values
            )
            dst_conn.commit()
            cur.close()
            values = []
        dt_aux += dt.timedelta(hours=1)
    if values:
        logger.info(f"Inserted {i * len(values) + len(values)} date times")
        cur = dst_conn.cursor()
        cur.executemany("""
            insert into dim_time (
                dt, year, month, day, hour, quarter, day_name, day_name_abbr, month_name, month_name_abbr, is_weekend, day_period
            ) values (
                :dt, :year, :month, :day, :hour, :quarter, :day_name, :day_name_abbr, :month_name, :month_name_abbr, :is_weekend, :day_period
            )
            """,
            values
        )
        dst_conn.commit()
        cur.close()
        values = []

In [13]:
def insert_dim_product(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            p.ProductID as product_id,
            p.ProductName as product_name,
            c.CategoryName as category_name,
            c.Description as category_description,
            s.CompanyName as supplier_company,
            s.City as supplier_city,
            s.Region as supplier_region,
            s.Country as supplier_country,
            p.UnitPrice as product_unit_price,
            p.QuantityPerUnit as product_quantity_per_unit,
            case
                when p.Discontinued = '0'
                    then false
                when p.Discontinued = '1'
                    then true
                else false
            end as product_discontinued
        from Products p
        inner join Categories c on p.CategoryID = c.CategoryID
        inner join Suppliers s on p.SupplierID = s.SupplierID
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_product (
            product_id, product_name, category_name, category_description, supplier_company, supplier_city, supplier_region, supplier_country, product_unit_price,
            product_quantity_per_unit, product_discontinued
        ) values (
            :product_id, :product_name, :category_name, :category_description, :supplier_company, :supplier_city, :supplier_region, :supplier_country, :product_unit_price,
            :product_quantity_per_unit, :product_discontinued
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [14]:
def insert_dim_customer(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            CustomerID as customer_id,
            CompanyName as company_name,
            ContactName as contact_name,
            City as city,
            Region as region,
            Country as country
        from Customers
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_customer (
            customer_id, company_name, contact_name, city, region, country
        ) values (
            :customer_id, :company_name, :contact_name, :city, :region, :country
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [15]:
def insert_dim_employee(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            e1.EmployeeID as employee_id,
            e1.FirstName || ' ' || e1.LastName as full_name,
            e1.Title as title,
            datetime(e1.BirthDate) as birth_date,
            datetime(e1.HireDate) as hire_date,
            e1.City as city,
            e1.Region as region,
            e1.Country as country,
            e2.FirstName || ' ' || e2.LastName as reports_to_name
        from Employees e1
            left outer join Employees e2 on e1.ReportsTo = e2.EmployeeID
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_employee (
            employee_id, full_name, title, birth_date, hire_date, city,
            region, country, reports_to_name
        ) values (
            :employee_id, :full_name, :title, :birth_date, :hire_date, :city,
            :region, :country, :reports_to_name
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [16]:
def insert_dim_shipper(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            ShipperID as shipper_id,
            CompanyName as company_name
        from Shippers
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_shipper (
            shipper_id, company_name
        ) values (
            :shipper_id, :company_name
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [17]:
def insert_dim_ship_location(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select distinct
            ShipCity as city,
            ShipRegion as region,
            ShipCountry as country
        from Orders
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_ship_location (
            city, region, country
        ) values (
            :city, :region, :country
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [24]:
def insert_fact_sales(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            o.OrderID as order_id,
            datetime(o.OrderDate) as order_dt,
            datetime(o.RequiredDate) as required_dt,
            datetime(o.ShippedDate) as shipped_dt,
            o.CustomerID as customer_id,
            od.ProductID as product_id,
            o.EmployeeID as employee_id,
            o.ShipVia as shipper_id,
            o.ShipCity as ship_city,
            o.ShipRegion as ship_region,
            o.ShipCountry as ship_country,
            od.UnitPrice as unity_price_sold,
            od.Quantity as quantity,
            od.Discount as discount,
            round((od.UnitPrice * od.Quantity) * (1 - od.Discount), 2) as revenue_net,
            0 as freight_apportioned
        from Orders o
        inner join main."Order Details" od on o.OrderID = od.OrderID
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    dst_cur = dst_conn.cursor()
    final_values = []
    i = 0
    for value in values:
        if len(final_values) > 0 and len(final_values) % 10_000 == 0:
            i += 1
            logger.info(f"{i}: Inserted fact values")
            cur = dst_conn.cursor()
            cur.executemany("""
                insert into fact_sales (
                    order_id,
                    order_time_id,
                    required_time_id,
                    shipped_time_id,
                    customer_id,
                    product_id,
                    employee_id,
                    shipper_id,
                    ship_location_id,
                    unit_price_sold,
                    quantity,
                    discount,
                    revenue_net,
                    freight_apportioned
                ) values (
                    ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
                )
                """,
                final_values
            )
            dst_conn.commit()
            cur.close()
            final_values = []
        # Operações Lookup
        order_time_id = get_time_id(dst_cur, value["order_dt"])
        required_time_id = get_time_id(dst_cur, value["required_dt"])
        shipped_time_id = get_time_id(dst_cur, value["shipped_dt"])
        customer_id = dst_cur.execute(
            "select id from dim_customer where customer_id = ?", (value["customer_id"],)
        ).fetchone()
        product_id = dst_cur.execute(
            "select id from dim_product where product_id = ?", (value["product_id"],)
        ).fetchone()
        employee_id = dst_cur.execute(
            "select id from dim_employee where employee_id = ?", (value["employee_id"],)
        ).fetchone()
        shipper_id = dst_cur.execute(
            "select id from dim_shipper where shipper_id = ?", (value["shipper_id"],)
        ).fetchone()
        ship_location_id = dst_cur.execute(
            "select id from dim_ship_location where city = ? and region = ? and country = ?", (value["ship_city"], value["ship_region"], value["ship_country"])
        ).fetchone()
        final_values.append(
            (
                value["order_id"],
                order_time_id,
                required_time_id,
                shipped_time_id,
                customer_id["id"],
                product_id["id"],
                employee_id["id"],
                shipper_id["id"],
                ship_location_id["id"],
                value["unity_price_sold"],
                value["quantity"],
                value["discount"],
                value["revenue_net"],
                0
            )
        )
    if final_values:
        logger.info(f"{i}: Inserted fact values")
        cur = dst_conn.cursor()
        cur.executemany("""
            insert into fact_sales (
                order_id,
                order_time_id,
                required_time_id,
                shipped_time_id,
                customer_id,
                product_id,
                employee_id,
                shipper_id,
                ship_location_id,
                unit_price_sold,
                quantity,
                discount,
                revenue_net,
                freight_apportioned
            ) values (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """,
            final_values
        )
        dst_conn.commit()
        cur.close()
        final_values = []
    dst_conn.commit()
    dst_cur.close()

In [25]:
with sqlite3.connect("northwind.db") as oltp_conn, sqlite3.connect("northwind-dw.db") as olap_conn:
    oltp_conn.row_factory = dict_factory
    olap_conn.row_factory = dict_factory
    # DDL
    create_dim_time(olap_conn)
    create_dim_product(olap_conn)
    create_dim_customer(olap_conn)
    create_dim_employee(olap_conn)
    create_dim_shipper(olap_conn)
    create_dim_ship_location(olap_conn)
    create_fact_sales(olap_conn)
    # ETL
    # insert_dim_time(oltp_conn, olap_conn)
    # insert_dim_product(oltp_conn, olap_conn)
    # insert_dim_customer(oltp_conn, olap_conn)
    # insert_dim_employee(oltp_conn, olap_conn)
    # insert_dim_shipper(oltp_conn, olap_conn)
    # insert_dim_ship_location(oltp_conn, olap_conn)
    insert_fact_sales(oltp_conn, olap_conn)

INFO:__main__:1: Inserted fact values
INFO:__main__:2: Inserted fact values
INFO:__main__:3: Inserted fact values
INFO:__main__:4: Inserted fact values
INFO:__main__:5: Inserted fact values
INFO:__main__:6: Inserted fact values
INFO:__main__:7: Inserted fact values
INFO:__main__:8: Inserted fact values
INFO:__main__:9: Inserted fact values
INFO:__main__:10: Inserted fact values
INFO:__main__:11: Inserted fact values
INFO:__main__:12: Inserted fact values
INFO:__main__:13: Inserted fact values
INFO:__main__:14: Inserted fact values
INFO:__main__:15: Inserted fact values
INFO:__main__:16: Inserted fact values
INFO:__main__:17: Inserted fact values
INFO:__main__:18: Inserted fact values
INFO:__main__:19: Inserted fact values
INFO:__main__:20: Inserted fact values
INFO:__main__:21: Inserted fact values
INFO:__main__:22: Inserted fact values
INFO:__main__:23: Inserted fact values
INFO:__main__:24: Inserted fact values
INFO:__main__:25: Inserted fact values
INFO:__main__:26: Inserted fact va